### PyTorch Tensors:A Practical Guide
**By Ananda Rimal**

*From tensor creation to mathematical operations, copying, and autograd.*

> **How to use:** Run the cells from top to bottom. Each example is short and includes the main idea you should remember.

### What you will learn
1. Create tensors and control random values.
2. Inspect shape, dtype, and device.
3. Perform scalar, element-wise, reduction, and matrix operations.
4. Understand in-place updates and real copies.
5. Use indexing, reshaping, broadcasting, and basic gradients.

---
## 1. Setup
A **tensor** stores numbers in one or more dimensions. PyTorch uses tensors for data, neural-network weights, and calculations.

In [1]:
import torch

# PyTorch version
torch.__version__

'2.10.0+cpu'

In [2]:
# CUDA available
torch.cuda.is_available()

False

In [3]:
# Device
torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

'CPU'

---
## 2. Creating Tensors
The numbers in `size=(2, 3)` mean **2 rows × 3 columns = 6 elements**.

### 2.1 `empty`, `zeros`, and `ones`
`empty()` allocates memory **without setting starting values**. Its displayed numbers are not reliable data.

In [4]:
empty_tensor = torch.empty(2, 3)
zeros_tensor = torch.zeros(2, 3)
ones_tensor = torch.ones(2, 3)

# empty (uninitialized)
empty_tensor

tensor([[2.8248e-34, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 5.3968e+28, 4.1777e-41]])

In [5]:
# zeros
zeros_tensor

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [6]:
# ones
ones_tensor

tensor([[1., 1., 1.],
        [1., 1., 1.]])

> **Important:** `torch.empty()` is not a tensor with zero elements. Its values are *uninitialized*. Fill them before doing meaningful calculations.

In [7]:
filled = torch.empty(2, 3)
filled.fill_(5)  # Set every element to 5 (in-place)

filled

tensor([[5., 5., 5.],
        [5., 5., 5.]])

### 2.2 Random tensors
- `rand()` gives floating-point values in **[0, 1)**.
- `randn()` gives values from a standard normal distribution.
- `randint(low, high, size)` gives integers from **low inclusive** to **high exclusive**.

In [8]:
# rand
torch.rand(2, 3)

tensor([[0.7690, 0.2621, 0.0821],
        [0.8950, 0.0567, 0.0301]])

In [9]:
# randn
torch.randn(2, 3)

tensor([[ 0.5177,  0.0662,  0.2970],
        [-0.5739,  0.3380, -0.3236]])

In [10]:
# randint [0, 10)
torch.randint(low=0, high=10, size=(2, 3))

tensor([[2, 9, 3],
        [1, 8, 9]])

### 2.3 Random seed
`torch.manual_seed(100)` resets the random generator. Reset to the **same seed** before identical operations to reproduce the same numbers in the same environment.

In [11]:
torch.manual_seed(100)
first = torch.rand(2, 3)

torch.manual_seed(100)
second = torch.rand(2, 3)

# First
first

tensor([[0.1117, 0.8158, 0.2626],
        [0.4839, 0.6765, 0.7539]])

In [12]:
# Second
second

tensor([[0.1117, 0.8158, 0.2626],
        [0.4839, 0.6765, 0.7539]])

In [13]:
# Same?
torch.equal(first, second)

True

In [14]:
torch.manual_seed(100)
a = torch.rand(2, 3)
b = torch.rand(2, 3)  # Generator advances, so this is different

# First call
a

tensor([[0.1117, 0.8158, 0.2626],
        [0.4839, 0.6765, 0.7539]])

In [15]:
# Second call
b

tensor([[0.2627, 0.0428, 0.2080],
        [0.1180, 0.1217, 0.7356]])

### 2.4 Create a tensor from Python data
`torch.tensor()` converts Python lists (or nested lists) into a tensor.

In [16]:
x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])

x

tensor([[1, 2, 3],
        [4, 5, 6]])

In [17]:
# Type
type(x)

torch.Tensor

### 2.5 Other useful creation functions
`arange` uses a step; `linspace` uses a number of points; `eye` makes an identity matrix; `full` fills every element with a chosen value.

In [18]:
# arange
torch.arange(0, 10, 2)

tensor([0, 2, 4, 6, 8])

In [19]:
# linspace
torch.linspace(0, 10, steps=5)

tensor([ 0.0000,  2.5000,  5.0000,  7.5000, 10.0000])

In [20]:
# identity
torch.eye(3)

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])

In [21]:
# full
torch.full((2, 3), 7)

tensor([[7, 7, 7],
        [7, 7, 7]])

### 2.6 Create a tensor like another tensor
These functions reuse the **shape** of `x`. Some also reuse its dtype.

In [22]:
# x
x

tensor([[1, 2, 3],
        [4, 5, 6]])

In [23]:
# empty_like (uninitialized)
torch.empty_like(x)

tensor([[7364621487442440760, 3904729944292604001, 7149806796351747429],
        [4123388941958723381, 3689121434962048309, 7365137132709700660]])

In [24]:
# zeros_like
torch.zeros_like(x)

tensor([[0, 0, 0],
        [0, 0, 0]])

In [25]:
# ones_like
torch.ones_like(x)

tensor([[1, 1, 1],
        [1, 1, 1]])

In [26]:
# rand_like (float)
torch.rand_like(x, dtype=torch.float32)

tensor([[0.7118, 0.7876, 0.4183],
        [0.9014, 0.9969, 0.7565]])

---
## 3. Shapes, Dimensions, and Data Types
For a 2D tensor, the shape is `(rows, columns)`. `ndim` is the number of axes; `numel()` is the number of values.

In [27]:
x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])

# Shape
x.shape

torch.Size([2, 3])

In [28]:
# Dimensions
x.ndim

2

In [29]:
# Elements
x.numel()

6

In [30]:
# dtype
x.dtype

torch.int64

In [31]:
# device
x.device

device(type='cpu')

### 3.1 Common dtypes
| dtype | Stores | Common use |
|---|---|---|
| `torch.float32` | 32-bit decimals | Model weights and training |
| `torch.float64` | 64-bit decimals | Higher precision |
| `torch.float16` / `torch.bfloat16` | 16-bit decimals | Lower-memory training/inference |
| `torch.int64` | 64-bit integers | Integer labels and indices |
| `torch.int32` | 32-bit integers | Integer calculations |
| `torch.uint8` | Unsigned 8-bit integers | Image pixels |
| `torch.bool` | True/False | Masks and comparisons |

Other types, including complex and quantized dtypes, are available for specialized tasks.

### 3.2 Set or convert a dtype
`.to(dtype)` normally returns another tensor; **it does not change `x` in-place**.

In [32]:
integer_tensor = torch.tensor([1, 2, 3], dtype=torch.int32)
float_tensor = torch.tensor([1, 2, 3], dtype=torch.float64)

(integer_tensor, integer_tensor.dtype)

(tensor([1, 2, 3], dtype=torch.int32), torch.int32)

In [33]:
(float_tensor, float_tensor.dtype)

(tensor([1., 2., 3.], dtype=torch.float64), torch.float64)

In [34]:
x_float = x.to(torch.float32)

# Original x
x.dtype

torch.int64

In [35]:
# Converted x_float
x_float.dtype

torch.float32

> **Tip:** A PyTorch tensor has a single dtype for its elements. Neural-network computations typically use floating-point tensors.

---
## 4. Tensor Mathematical Operations
We will use small tensors so each result is easy to inspect.

### 4.1 Scalar operations
A scalar is one number. Adding or multiplying a tensor by a scalar applies the operation to **every element**.

In [36]:
x = torch.tensor([[2., 4.],
                  [6., 8.]])

# x
x

tensor([[2., 4.],
        [6., 8.]])

In [37]:
# x + 2
x + 2

tensor([[ 4.,  6.],
        [ 8., 10.]])

In [38]:
# x - 2
x - 2

tensor([[0., 2.],
        [4., 6.]])

In [39]:
# x * 3
x * 3

tensor([[ 6., 12.],
        [18., 24.]])

In [40]:
# x / 2
x / 2

tensor([[1., 2.],
        [3., 4.]])

In [41]:
# x // 3
x // 3

tensor([[0., 1.],
        [2., 2.]])

In [42]:
# x % 3
x % 3

tensor([[2., 1.],
        [0., 2.]])

In [43]:
# x ** 2
x ** 2

tensor([[ 4., 16.],
        [36., 64.]])

### 4.2 Element-wise operations
For two tensors of the same shape, values at matching positions are combined. `*` means **element-wise multiplication**, not matrix multiplication.

In [44]:
a = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
b = torch.tensor([[2., 2., 2.],
                  [2., 2., 2.]])

# a + b
a + b

tensor([[3., 4., 5.],
        [6., 7., 8.]])

In [45]:
# a - b
a - b

tensor([[-1.,  0.,  1.],
        [ 2.,  3.,  4.]])

In [46]:
# a * b
a * b

tensor([[ 2.,  4.,  6.],
        [ 8., 10., 12.]])

In [47]:
# a / b
a / b

tensor([[0.5000, 1.0000, 1.5000],
        [2.0000, 2.5000, 3.0000]])

In [48]:
# a ** b
a ** b

tensor([[ 1.,  4.,  9.],
        [16., 25., 36.]])

In [49]:
# a % b
a % b

tensor([[1., 0., 1.],
        [0., 1., 0.]])

### 4.3 Absolute value, rounding, and clamping
Useful for preprocessing data and restricting values.

In [50]:
c = torch.tensor([1, -2, 3, -4])
d = torch.tensor([1.9, 2.3, 3.7, 4.4])

# abs
torch.abs(c)

tensor([1, 2, 3, 4])

In [51]:
# neg
torch.neg(c)

tensor([-1,  2, -3,  4])

In [52]:
# round
torch.round(d)

tensor([2., 2., 4., 4.])

In [53]:
# ceil
torch.ceil(d)

tensor([2., 3., 4., 5.])

In [54]:
# floor
torch.floor(d)

tensor([1., 2., 3., 4.])

In [55]:
# clamp between 2 and 3
torch.clamp(d, min=2, max=3)

tensor([2.0000, 2.3000, 3.0000, 3.0000])

### 4.4 Reduction operations
Reductions combine multiple elements into fewer values. For a 2D tensor:
- `dim=0` reduces **down the rows**, giving one result per column.
- `dim=1` reduces **across the columns**, giving one result per row.

In [56]:
e = torch.tensor([[1., 2., 9.],
                  [7., 6., 5.]])

# e
e

tensor([[1., 2., 9.],
        [7., 6., 5.]])

In [57]:
# Total sum
torch.sum(e)

tensor(30.)

In [58]:
# dim=0 (column totals)
torch.sum(e, dim=0)

tensor([ 8.,  8., 14.])

In [59]:
# dim=1 (row totals)
torch.sum(e, dim=1)

tensor([12., 18.])

In [60]:
# Mean
torch.mean(e)

tensor(5.)

In [61]:
# Mean, dim=0
torch.mean(e, dim=0)

tensor([4., 4., 7.])

In [62]:
# Median
torch.median(e)

tensor(5.)

In [63]:
# Min
torch.min(e)

tensor(1.)

In [64]:
# Max
torch.max(e)

tensor(9.)

In [65]:
# Product
torch.prod(e)

tensor(3780.)

In [66]:
# Std (sample)
torch.std(e)

tensor(3.0332)

In [67]:
# Variance (sample)
torch.var(e)

tensor(9.2000)

In [68]:
# argmax (flat index)
torch.argmax(e)

tensor(2)

In [69]:
# argmin (flat index)
torch.argmin(e)

tensor(0)

> **Note:** By default, `torch.std()` and `torch.var()` use the sample correction (`correction=1`). Use `correction=0` for population statistics.

### 4.5 Matrix multiplication
If `f` has shape `(2, 3)` and `g` has shape `(3, 2)`, then `f @ g` has shape `(2, 2)`. The two **inner dimensions must match**.

In [70]:
f = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
g = torch.tensor([[7, 8],
                  [9, 10],
                  [11, 12]])

# f shape
f.shape

torch.Size([2, 3])

In [71]:
# g shape
g.shape

torch.Size([3, 2])

In [72]:
# f @ g
torch.matmul(f, g)

tensor([[ 58,  64],
        [139, 154]])

In [73]:
# Result shape
(f @ g).shape

torch.Size([2, 2])

### 4.6 Dot product and transpose
A dot product of two 1D vectors returns a scalar. A transpose swaps selected axes.

In [74]:
vector1 = torch.tensor([1, 2])
vector2 = torch.tensor([3, 4])

# Dot product
torch.dot(vector1, vector2)

tensor(11)

In [75]:
# Transpose of f
torch.transpose(f, 0, 1)

tensor([[1, 4],
        [2, 5],
        [3, 6]])

In [76]:
# Using .T
f.T

tensor([[1, 4],
        [2, 5],
        [3, 6]])

### 4.7 Determinant and inverse
An inverse exists only for a **square, nonsingular** matrix. We use a fixed invertible matrix rather than a random one.

In [77]:
h = torch.tensor([[2., 1.],
                  [1., 1.]])

# Determinant
torch.linalg.det(h)

tensor(1.)

In [78]:
# Inverse
torch.linalg.inv(h)

tensor([[ 1., -1.],
        [-1.,  2.]])

In [79]:
# h @ inverse(h)
h @ torch.linalg.inv(h)

tensor([[1., 0.],
        [0., 1.]])

### 4.8 Comparison operations
Comparisons return a tensor of Boolean values (`True` / `False`).

In [80]:
i = torch.tensor([[1, 5, 3], [7, 2, 9]])
j = torch.tensor([[2, 5, 1], [6, 4, 9]])

# i > j
i > j

tensor([[False, False,  True],
        [ True, False, False]])

In [81]:
# i < j
i < j

tensor([[ True, False, False],
        [False,  True, False]])

In [82]:
# i == j
i == j

tensor([[False,  True, False],
        [False, False,  True]])

In [83]:
# i != j
i != j

tensor([[ True, False,  True],
        [ True,  True, False]])

In [84]:
# i >= j
i >= j

tensor([[False,  True,  True],
        [ True, False,  True]])

In [85]:
# i <= j
i <= j

tensor([[ True,  True, False],
        [False,  True,  True]])

### 4.9 Special mathematical functions
`log` needs positive values for a finite real-valued result; `sqrt` needs nonnegative values.

In [86]:
k = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])

# log
torch.log(k)

tensor([[0.0000, 0.6931, 1.0986],
        [1.3863, 1.6094, 1.7918]])

In [87]:
# exp
torch.exp(k)

tensor([[  2.7183,   7.3891,  20.0855],
        [ 54.5981, 148.4132, 403.4288]])

In [88]:
# sqrt
torch.sqrt(k)

tensor([[1.0000, 1.4142, 1.7321],
        [2.0000, 2.2361, 2.4495]])

In [89]:
# sigmoid
torch.sigmoid(k)

tensor([[0.7311, 0.8808, 0.9526],
        [0.9820, 0.9933, 0.9975]])

In [90]:
# ReLU
torch.relu(k - 4)

tensor([[0., 0., 0.],
        [0., 1., 2.]])

**Softmax** turns values into probabilities. For class scores arranged in rows, `dim=1` makes **each row sum to 1**.

In [91]:
logits = torch.tensor([[2., 1., 0.],
                       [0., 1., 2.]])
probabilities = torch.softmax(logits, dim=1)

# Softmax probabilities
probabilities

tensor([[0.6652, 0.2447, 0.0900],
        [0.0900, 0.2447, 0.6652]])

In [92]:
# Row sums
probabilities.sum(dim=1)

tensor([1., 1.])

---
## 5. In-Place Operations
An underscore at the end of a PyTorch method, such as `.add_()` or `.relu_()`, usually means it **modifies the existing tensor**.

### 5.1 `add()` vs `add_()`
`m.add(n)` returns a result without changing `m`; `m.add_(n)` changes `m` in-place.

In [93]:
m = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])
n = torch.tensor([[10., 20., 30.],
                  [40., 50., 60.]])

result = m.add(n)

# Result of m.add(n)
result

tensor([[11., 22., 33.],
        [44., 55., 66.]])

In [94]:
# m after add (unchanged)
m

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [95]:
m.add_(n)

# m after add_ (modified)
m

tensor([[11., 22., 33.],
        [44., 55., 66.]])

### 5.2 In-place ReLU
`relu_()` changes the original tensor; `relu()` returns a result.

In [96]:
values = torch.tensor([[-2., 3.],
                       [4., -5.]])

# relu(values)
torch.relu(values)

tensor([[0., 3.],
        [4., 0.]])

In [97]:
# values before relu_
values

tensor([[-2.,  3.],
        [ 4., -5.]])

In [98]:
values.relu_()

# values after relu_
values

tensor([[0., 3.],
        [4., 0.]])

> **Remember:** In-place updates persist in the tensor object until changed again. They do **not** automatically save anything to disk. Some in-place operations can interfere with autograd; use them carefully during training.

---
## 6. Assignment vs Copying a Tensor
The distinction is similar to Pandas: assignment creates **another reference**, while cloning creates **separate storage**.

### 6.1 `b = a` — same object
Changing `b` in-place also changes `a`, because both names point to the same tensor.

In [99]:
a = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
b = a

# Same Python object?
a is b

True

In [100]:
b[0, 0] = 100

# a
a

tensor([[100,   2,   3],
        [  4,   5,   6]])

In [101]:
# b
b

tensor([[100,   2,   3],
        [  4,   5,   6]])

### 6.2 `b = a.clone()` — separate copy
Cloning allocates separate tensor storage. Editing `b` no longer changes `a`.

In [102]:
a = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
b = a.clone()

# Same Python object?
a is b

False

In [103]:
# Same memory address?
a.data_ptr() == b.data_ptr()

False

In [104]:
b[0, 0] = 100

# a (unchanged)
a

tensor([[1, 2, 3],
        [4, 5, 6]])

In [105]:
# b (changed)
b

tensor([[100,   2,   3],
        [  4,   5,   6]])

### 6.3 Reassignment is not an in-place update
`b = b + 10` makes `b` refer to a new tensor. By contrast, `b.add_(10)` edits shared storage.

In [106]:
a = torch.tensor([1, 2, 3])
b = a
b = b + 10

# After b = b + 10
{'a =': a, 'b =': b}

{'a =': tensor([1, 2, 3]), 'b =': tensor([11, 12, 13])}

In [107]:
b = a
b.add_(10)

# After b.add_(10)
{'a =': a, 'b =': b}

{'a =': tensor([11, 12, 13]), 'b =': tensor([11, 12, 13])}

| Expression | What happens? |
|---|---|
| `b = a` | Same tensor object |
| `b = a.clone()` | Independent tensor storage |
| `b = a.detach()` | New tensor sharing storage; gradient history detached |
| `b = a.detach().clone()` | Independent storage; no gradient connection |
| `a.add_(1)` | Modifies `a` in-place |

**Pandas analogy:** `df2 = df1` shares a DataFrame object, while `df2 = df1.copy()` creates a separate DataFrame.

---
## 7. Indexing and Slicing
Use `[row, column]` to select one element; `:` selects a range of positions.

In [108]:
x = torch.tensor([[10, 20, 30],
                  [40, 50, 60]])

# Element [0,1]
x[0, 1]

tensor(20)

In [109]:
# First row
x[0]

tensor([10, 20, 30])

In [110]:
# Second column
x[:, 1]

tensor([20, 50])

In [111]:
# First two columns
x[:, :2]

tensor([[10, 20],
        [40, 50]])

In [112]:
# Values above 30
x[x > 30]

tensor([40, 50, 60])

> **Important:** Basic slicing often returns a **view** sharing storage with the original tensor. Modify it in-place only when you intend to affect the original.

---
## 8. Reshaping and Combining Tensors
Reshaping changes dimensions without changing the total number of elements.

In [113]:
x = torch.arange(12)

# Original
(x.shape, x)

(torch.Size([12]), tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]))

In [114]:
# reshape(3,4)
x.reshape(3, 4)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

In [115]:
# reshape(2,6)
x.reshape(2, 6)

tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])

In [116]:
# flatten
x.reshape(3, 4).flatten()

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

`reshape()` may produce a view or a copy depending on the tensor layout. Use `clone()` when you explicitly need independent storage.

In [117]:
x = torch.tensor([1, 2, 3])

# Original shape
x.shape

torch.Size([3])

In [118]:
# unsqueeze(0)
x.unsqueeze(0).shape

torch.Size([1, 3])

In [119]:
# unsqueeze(1)
x.unsqueeze(1).shape

torch.Size([3, 1])

In [120]:
# squeeze back
x.unsqueeze(0).squeeze(0).shape

torch.Size([3])

### Concatenation vs stacking
`cat` joins tensors along an existing axis. `stack` adds a **new axis**.

In [121]:
p = torch.tensor([1, 2])
q = torch.tensor([3, 4])

# cat
torch.cat((p, q), dim=0)

tensor([1, 2, 3, 4])

In [122]:
# stack
torch.stack((p, q), dim=0)

tensor([[1, 2],
        [3, 4]])

In [123]:
# stack shape
torch.stack((p, q), dim=0).shape

torch.Size([2, 2])

---
## 9. Broadcasting
PyTorch can apply element-wise operations to certain different shapes when their dimensions are **compatible**.

In [124]:
matrix = torch.tensor([[1., 2., 3.],
                       [4., 5., 6.]])  # (2, 3)
row = torch.tensor([10., 20., 30.])     # (3,)

# matrix + row
matrix + row

tensor([[11., 22., 33.],
        [14., 25., 36.]])

In [125]:
# Result shape
(matrix + row).shape

torch.Size([2, 3])

**Rule:** Compare dimensions from the right. Each pair must be equal, or one dimension must be `1`, or a dimension must be missing.

---
## 10. CPU and GPU
A tensor has a `device`. Use `.to(device)` to move data to CPU or GPU, if available.

In [126]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.tensor([1., 2., 3.])
x_device = x.to(device)

# Selected device
device

device(type='cpu')

In [127]:
# Tensor device
x_device.device

device(type='cpu')

> **Important:** Tensors used in the same ordinary operation generally need to be on the **same device**.

---
## 11. First Look at Autograd
PyTorch can automatically calculate derivatives for floating-point tensors with `requires_grad=True`. This is the basis of neural-network backpropagation.

In [128]:
w = torch.tensor(3.0, requires_grad=True)
y = w ** 2 + 2 * w    # y = w² + 2w

y.backward()        # dy/dw = 2w + 2

# w
w.item()

3.0

In [129]:
# y
y.item()

15.0

In [130]:
# dy/dw
w.grad.item()

8.0

**Gradient note:** Gradients accumulate in `.grad` on repeated backward passes. During training, optimizers typically clear old gradients before the next step.

---
## 12. Quick Practice
Try these without looking back:

1. Create a `(3, 4)` tensor filled with `0`.
2. Generate random integers from `0` to `9` inclusive in a `(2, 3)` tensor.
3. Create a `(2, 3)` tensor, then find its shape, dtype, and number of elements.
4. Add two `(2, 3)` tensors element-wise.
5. Multiply a `(2, 3)` matrix with a `(3, 2)` matrix.
6. Explain why `b = a` is different from `b = a.clone()`.
7. Find the sum across columns using `dim=0`.
8. Compute the gradient of `y = x**2` at `x=4`.

**Core takeaway:** `=` assigns a reference, `.clone()` makes an independent tensor, and an underscore method like `.add_()` modifies the existing tensor.